In [1]:
import torchvision
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import numpy as np

import ModelFxp12 as fxp

In [2]:
class GetChannel:
    def __init__(self, channel):
      self.channel = channel

    def __call__(self, img):
        return  torch.unsqueeze(img[self.channel], 0)

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.3403, 0.3121, 0.3214),
                         (0.2724, 0.2608, 0.2669)),
    #transforms.Grayscale(),
    GetChannel(1)
])

# Create Datasets
testset = torchvision.datasets.GTSRB(root='german-traffic-sign', split='test', download=True, transform=transform)

# Load Datasets
test_loader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=True)

100%|██████████| 88978620/88978620 [00:05<00:00, 17486143.31it/s]


Extracting german-traffic-sign/gtsrb/GTSRB_Final_Test_Images.zip to german-traffic-sign/gtsrb


100%|██████████| 99620/99620 [00:00<00:00, 242236.45it/s]


Extracting german-traffic-sign/gtsrb/GTSRB_Final_Test_GT.zip to german-traffic-sign/gtsrb


In [3]:
class NetGTSRB(nn.Module):
    def __init__(self, num_classes):
        nn.Module.__init__(self)
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=16, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout(p=0.55)
        )
        self.fc = nn.Sequential(
            nn.Linear(in_features=16 * 6 * 6, out_features=num_classes) # FC output 5 classes
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        #print(x.shape)
        x = x.view(x.size()[0], -1)
        x = self.fc(x)
        return x

In [4]:
# Computes the average accuracy
def evaluate_fxp(model, val_loader):

    batch_accuracies = []   # Accuracies of each forwarded batch

    for images, labels in val_loader:
        outputs = model(images)                    # Generates batch predictions

        # Computes the batch accuracy
        _, preds = torch.max(torch.from_numpy(outputs), dim=1)
        accuracy = torch.tensor(torch.sum(preds == labels).item() / len(preds))

        # Append the batch accuracy
        batch_accuracies.append(accuracy)

    # Computes the epoch average accuracy
    accuracy = torch.stack(batch_accuracies).mean() # Average accuracy

    return accuracy

In [5]:
# Create the fixed point model
def model_fxp(model, shift):

    layers = [
        # conv1
        fxp.Conv(model.conv1[0], shift, shift),
        fxp.ReLU(),
        fxp.MaxPool2d(2),

        # conv2
        fxp.Conv(model.conv2[0], shift, shift),
        fxp.ReLU(),
        fxp.MaxPool2d(2),

        # fc1
        fxp.Linear(model.fc[0], shift, shift)
    ]

    return fxp.ModelFxp(layers)

In [ ]:
# Load the quantized floating point model
model = NetGTSRB(43)
model.load_state_dict(torch.load('train2_qat_sh5.pt', map_location=torch.device('cpu')))

# Report (GetChannel(1))
for shift in (range(5,10)):
    print(f"Shift: {shift}, accuracy: {evaluate_fxp(model_fxp(model, shift), test_loader)}\n")

#     Parameters          shift   accuracy
#  train2_qat_sh2.pt      2       66.3
#  train2_qat_sh2.pt      3       70.6
#  train2_qat_sh2.pt      4       72.25

#  train2_qat_sh3.pt      3       88.63
#  train2_qat_sh3.pt      4       91.17
#  train2_qat_sh3.pt      5       92.63
#  train2_qat_sh3.pt      6       92.86
#  train2_qat_sh3.pt      7       92.13
#  train2_qat_sh3.pt      8       92.80

#  train2_qat_sh4.pt      4       93.19
#  train2_qat_sh4.pt      5       93.9
#  train2_qat_sh4.pt      6       93.96
#  train2_qat_sh4.pt      7       93.87
#  train2_qat_sh4.pt      8       93.94
#  train2_qat_sh4.pt      9       93.95

#  train2_qat_sh5.pt      5       94.18
#  train2_qat_sh5.pt      6       94.26
#  train2_qat_sh5.pt      7       94.27
#  train2_qat_sh5.pt      8       94.32
#  train2_qat_sh5.pt      9       94.31

#  train2_qat_sh6.pt      6       94.39
#  train2_qat_sh6.pt      7       94.45
#  train2_qat_sh6.pt      8       94.39
#  train2_qat_sh6.pt      9       94.44

#  train2_qat_sh7.pt      7       94.48
#  train2_qat_sh7.pt      8       94.44
#  train2_qat_sh7.pt      9       94.46


layers[0]: Conv
layers[1]: ReLU
layers[2]: MaxPool2d
layers[3]: Conv
layers[4]: ReLU
layers[5]: MaxPool2d
layers[6]: Linear
Shift: 5, accuracy: 0.9418824911117554

layers[0]: Conv
layers[1]: ReLU
layers[2]: MaxPool2d
layers[3]: Conv
layers[4]: ReLU
layers[5]: MaxPool2d
layers[6]: Linear


In [ ]:
model = NetGTSRB(43)
model.load_state_dict(torch.load('train2_qat_sh7.pt', map_location=torch.device('cpu')))

shift = 7
model_test = model_fxp(model, shift)
print(f"Shift: {shift}, accuracy: {evaluate_fxp(model_test, test_loader)}\n")


layers[0]: Conv
layers[1]: ReLU
layers[2]: MaxPool2d
layers[3]: Conv
layers[4]: ReLU
layers[5]: MaxPool2d
layers[6]: Linear
Shift: 7, accuracy: 0.9449601769447327



In [ ]:
model_test.AccumulatorsWidth()
model_test.WeightsWidth()

In [7]:
model = NetGTSRB(43)
model.load_state_dict(torch.load('train2_qat_sh7.pt', map_location=torch.device('cpu')))

shift = 7
model_test = model_fxp(model, shift)

layers[0]: Conv
layers[1]: ReLU
layers[2]: MaxPool2d
layers[3]: Conv
layers[4]: ReLU
layers[5]: MaxPool2d
layers[6]: Linear


In [8]:
print(model_test.ConvLayers[0].w.max())
print(model_test.ConvLayers[1].w.max())
print(model_test.FullLayers[0].w.max())

8
8
7
